# 数据可视化与验证（基于处理结果）

输入：`shaanxi_cities_temperature` 下的 1991-2023 与 2024 数据 / 样本 / 向量库结果。
本 Notebook 包含常用可视化：
- 数据摘要
- 年平均趋势
- 月平均气候态
- 单城市时间序列
- 随机 8 天逐小时对比

## 1. 参数与路径
根据需要调整 `DATA_DIR`（原始逐小时数据所在目录）。

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# 数据目录：指向 1991-2023 合并结果所在位置
DATA_DIR = Path('./shaanxi_cities_temperature')

# 查询城市列表（可修改）
CITIES = ['西安','郑州','太原','石家庄','济南','呼和浩特','银川','兰州','西宁','大同']

# 城市英文标签（用于图例）
CITY_LABELS = {
    '西安': "Xi'an",
    '郑州': 'Zhengzhou',
    '太原': 'Taiyuan',
    '石家庄': 'Shijiazhuang',
    '济南': 'Jinan',
    '呼和浩特': 'Hohhot',
    '银川': 'Yinchuan',
    '兰州': 'Lanzhou',
    '西宁': 'Xining',
    '大同': 'Datong',
}

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('DATA_DIR =', DATA_DIR)

DATA_DIR = shaanxi_cities_temperature


## 2. 工具函数

In [2]:
import numpy as np
import pandas as pd
import os
from datetime import datetime


def kelvin_to_celsius(temp_k):
    return temp_k - 273.15


def get_label(name: str) -> str:
    return CITY_LABELS.get(name, name)


def load_city_data(city_name):
    data_file = DATA_DIR / f"{city_name}_1991-2023.npy"
    time_file = DATA_DIR / f"{city_name}_1991-2023_times.npy"
    if not data_file.exists() or not time_file.exists():
        print(f"警告: {city_name} 的数据文件不存在")
        return None, None
    data = np.load(data_file)
    times = np.load(time_file, allow_pickle=True)
    return data, times

## 3. 数据摘要

In [3]:
def print_data_summary():
    print('='*80)
    print('陕西省主要城市 ERA5 2m 温度数据摘要')
    print('='*80)
    for city in CITIES:
        data, times = load_city_data(city)
        if data is None:
            continue
        data_c = kelvin_to_celsius(data)
        print(f"{city}:")
        print(f"  数据点数: {len(data)}")
        print(f"  时间范围: {times[0]} 至 {times[-1]}")
        print(f"  温度范围: {data_c.min():.2f}°C 至 {data_c.max():.2f}°C")
        print(f"  平均温度: {data_c.mean():.2f}°C")
        print(f"  标准差: {data_c.std():.2f}°C")

print_data_summary()

陕西省主要城市 ERA5 2m 温度数据摘要
西安:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -15.78°C 至 41.99°C
  平均温度: 15.11°C
  标准差: 10.30°C
郑州:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -13.15°C 至 42.14°C
  平均温度: 15.44°C
  标准差: 10.47°C
太原:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -24.10°C 至 38.86°C
  平均温度: 10.04°C
  标准差: 11.28°C
石家庄:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -15.66°C 至 43.58°C
  平均温度: 14.13°C
  标准差: 11.66°C
济南:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -19.10°C 至 40.22°C
  平均温度: 13.83°C
  标准差: 10.89°C
呼和浩特:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度范围: -34.86°C 至 36.73°C
  平均温度: 6.10°C
  标准差: 13.22°C
银川:
  数据点数: 289272
  时间范围: 1991-01-01T00:00:00.000000000 至 2023-12-31T23:00:00.000000000
  温度

## 4. 年平均温度趋势
生成 `annual_mean_temperature.png`

In [4]:
import matplotlib.pyplot as plt

def plot_annual_mean_temperature():
    fig, ax = plt.subplots(figsize=(14,8))
    for city in CITIES:
        data, times = load_city_data(city)
        if data is None:
            continue
        data_c = kelvin_to_celsius(data)
        times_dt = pd.to_datetime(times)
        df = pd.DataFrame({'time': times_dt, 'temp': data_c}).set_index('time')
        annual_mean = df.resample('YE').mean()
        ax.plot(annual_mean.index.year, annual_mean['temp'], marker='o', label=get_label(city), linewidth=2)
    ax.set_xlabel('Year'); ax.set_ylabel('Annual mean temperature (°C)')
    ax.set_title('Annual mean 2m temperature (1991-2023)')
    ax.legend(ncol=2); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out = DATA_DIR / 'annual_mean_temperature.png'
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print('已保存:', out)

plot_annual_mean_temperature()

已保存: shaanxi_cities_temperature/annual_mean_temperature.png


## 5. 月平均气候态
生成 `monthly_climatology.png`

In [5]:
import matplotlib.pyplot as plt

def plot_monthly_climatology():
    fig, ax = plt.subplots(figsize=(12,8))
    for city in CITIES:
        data, times = load_city_data(city)
        if data is None:
            continue
        data_c = kelvin_to_celsius(data)
        times_dt = pd.to_datetime(times)
        df = pd.DataFrame({'time': times_dt, 'temp': data_c}).set_index('time')
        monthly = df.groupby(df.index.month).mean()
        ax.plot(monthly.index, monthly['temp'], marker='o', label=get_label(city), linewidth=2)
    ax.set_xlabel('Month'); ax.set_ylabel('Mean temperature (°C)')
    ax.set_title('Monthly climatology (1991-2023)')
    ax.set_xticks(range(1,13)); ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    ax.legend(ncol=2); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out = DATA_DIR / 'monthly_climatology.png'
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print('已保存:', out)

plot_monthly_climatology()

已保存: shaanxi_cities_temperature/monthly_climatology.png


## 6. 单城市时间序列（可调年份范围）
设置 `CITY`, `START_YEAR`, `END_YEAR`，生成 `<city>_time_series.png`

In [6]:
CITY = '西安'          # 修改为任意城市
START_YEAR = None      # 例如 2000
END_YEAR = None        # 例如 2010

import matplotlib.pyplot as plt

def plot_city_time_series(city_name, start_year=None, end_year=None):
    data, times = load_city_data(city_name)
    if data is None:
        return
    data_c = kelvin_to_celsius(data)
    df = pd.DataFrame({'time': pd.to_datetime(times), 'temp': data_c}).set_index('time')
    if start_year and end_year:
        df = df[f'{start_year}':f'{end_year}']
    daily = df.resample('D').mean()
    monthly = df.resample('ME').mean()
    fig, ax = plt.subplots(figsize=(16,6))
    ax.plot(daily.index, daily['temp'], linewidth=0.5, alpha=0.7, label='Daily mean')
    ax.plot(monthly.index, monthly['temp'], 'r-', linewidth=2, label='Monthly mean')
    ax.set_xlabel('Time'); ax.set_ylabel('Temperature (°C)')
    ax.set_title(f"{get_label(city_name)} 2m temperature time series")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out = DATA_DIR / f"{city_name}_time_series.png"
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
    print('已保存:', out)

plot_city_time_series(CITY, START_YEAR, END_YEAR)

已保存: shaanxi_cities_temperature/西安_time_series.png


## 7. 随机 8 天逐小时对比
生成 `random_8day_hourly_*.png`

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_random_8day_hourly(num_samples=3, seed=None):
    if seed is not None:
        np.random.seed(seed)
    hourly_data = {}
    for city in CITIES:
        data, times = load_city_data(city)
        if data is None:
            continue
        df = pd.DataFrame({'time': pd.to_datetime(times), 'temp': kelvin_to_celsius(data)}).set_index('time').sort_index()
        hourly_data[city] = df
    if not hourly_data:
        print('无可用数据'); return
    # 选取范围交集
    min_ts = max(df.index.min() for df in hourly_data.values())
    max_ts = min(df.index.max() for df in hourly_data.values())
    if max_ts - min_ts < pd.Timedelta(days=8):
        print('数据时间范围不足 8 天'); return
    total_hours = int((max_ts - min_ts) / pd.Timedelta(hours=1))
    starts = np.random.randint(0, total_hours - 8*24, size=num_samples)
    for i, offset in enumerate(starts, start=1):
        t0 = min_ts + pd.Timedelta(hours=int(offset))
        t1 = t0 + pd.Timedelta(days=8)
        fig, ax = plt.subplots(figsize=(14,6))
        for city, df in hourly_data.items():
            seg = df.loc[t0:t1]
            if len(seg) < 8*24: continue
            hours = range(len(seg))
            ax.plot(hours, seg['temp'], label=get_label(city))
        ax.set_xlabel('Hour'); ax.set_ylabel('Temperature (°C)')
        ax.set_title(f'8-day hourly comparison #{i} (start {t0.date()})')
        ax.grid(True, alpha=0.3); ax.legend(ncol=2)
        plt.tight_layout()
        out = DATA_DIR / f'random_8day_hourly_{i}.png'
        plt.savefig(out, dpi=300, bbox_inches='tight'); plt.close()
        print('已保存:', out)

plot_random_8day_hourly(num_samples=3, seed=42)

已保存: shaanxi_cities_temperature/random_8day_hourly_1.png
已保存: shaanxi_cities_temperature/random_8day_hourly_2.png
已保存: shaanxi_cities_temperature/random_8day_hourly_3.png


## 8. 可视化输出清单

In [8]:
import glob
print('生成的图像：')
for f in sorted(glob.glob(str(DATA_DIR / '*.png'))):
    print(f)

生成的图像：
shaanxi_cities_temperature/annual_mean_temperature.png
shaanxi_cities_temperature/monthly_climatology.png
shaanxi_cities_temperature/random_8day_hourly_1.png
shaanxi_cities_temperature/random_8day_hourly_2.png
shaanxi_cities_temperature/random_8day_hourly_3.png
shaanxi_cities_temperature/西安_time_series.png
